Import Libraries and Mount Google Drive

In [ ]:
!pip install --upgrade mediapipe protobuf
!pip install --upgrade "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import glob
import cv2
import mediapipe as mp


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


Load IDs of missing videos from the dataset

In [ ]:

missing_path = "/content/drive/My Drive/Final Year Project/Data/missing.txt"
with open(missing_path, "r") as file:
    lines = file.readlines()
missing_ids = set([line.strip() for line in lines])



In [ ]:
len(missing_ids)

9103

Load Metadata (WLASL JSON)

In [ ]:
meta_data_path = "/content/drive/My Drive/Final Year Project/Data/WLASL_v0.3.json"
with open(meta_data_path) as file:
    meta_data = json.load(file)



Convert Meta-Data from json to df

In [ ]:
reference = pd.DataFrame()
all_data = []


for entry_index in range(len(meta_data)):
    gloss = meta_data[entry_index]['gloss']
    instances = meta_data[entry_index]['instances']

    for instance_dict in instances:
        # Check if the video ID should be excluded
        if instance_dict['video_id'] in missing_ids:
            continue # Skip this iteration and move to the next instance

        else:
            # Create a dictionary for the current instance's row
            row = {
                'gloss': gloss,
                'split': instance_dict['split'],
                'video_id': instance_dict['video_id'],
                'signer_id': instance_dict['signer_id'],
                'source': instance_dict['source'],
                'url': instance_dict['url'],
                'fps': instance_dict['fps'],
                'variation_id': instance_dict['variation_id'],

            }
            all_data.append(row)

reference = pd.DataFrame(all_data)



In [ ]:

reference.head()

,gloss,split,video_id,signer_id,source,url,fps,variation_id
0,book,train,69241,118,aslbrick,http://aslbricks.org/New/ASL-Videos/book.mp4,25,0
1,book,train,07069,31,signschool,https://signstock.blob.core.windows.net/signsc...,25,0
2,book,train,07068,36,startasl,https://s3-us-west-1.amazonaws.com/files.start...,25,0
3,book,train,07070,59,asldeafined,https://media.asldeafined.com/vocabulary/14666...,25,0
4,book,val,07099,12,aslsearch,http://www.aslsearch.com/signs/videos/book.mp4,25,0


In [ ]:
reference.fps.unique()

array([25])

In [ ]:
reference.variation_id.unique()

array([0, 1, 2])

Extract Keypoint from videos

In [ ]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

In [ ]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [ ]:
def draw_styled_landmarks(image, results):
    # Draw face connections
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
                              mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                              mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
                              )
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                              mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                              )
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                              mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                              )
    # Draw right hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                              mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                              )

In [ ]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() \
        if results.pose_landmarks else np.zeros(33*4)

    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() \
        if results.face_landmarks else np.zeros(468*3)

    left_hand = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() \
        if results.left_hand_landmarks else np.zeros(21*3)

    right_hand = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() \
        if results.right_hand_landmarks else np.zeros(21*3)

    return np.concatenate([pose, face, left_hand, right_hand])

In [ ]:
video_path = "/content/drive/My Drive/Final Year Project/Data/videos/"


files = [os.path.join(video_path, i) for i in os.listdir(video_path)]
print("Files:", files)

Files: ['/content/drive/My Drive/Final Year Project/Data/videos/65366.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65411.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65351.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65278.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65286.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65319.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65420.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65395.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65415.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65325.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65427.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65344.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65277.mp4', '/content/drive/My Drive/Final Year Project/Data/videos/65282.mp4', '/content/drive/My Drive/Final Year Proj

In [ ]:
print(len(files))

11980


In [ ]:
extensions = {os.path.splitext(f)[1] for f in files if os.path.isfile(f)}
print("Unique formats:", extensions)

Unique formats: {'.mp4'}


In [ ]:
def load_video_frames_uniform(video_path, sequence_length=30):
    """Load exactly `sequence_length` frames from a video using uniform sampling."""
    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        return None  # corrupted or unreadable video

    # Compute evenly spaced frame indices
    indices = np.linspace(0, total_frames - 1, sequence_length).astype(int)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            # If reading fails, repeat last good frame
            if len(frames) > 0:
                frames.append(frames[-1])
            continue

        frames.append(frame)

    cap.release()
    return frames


In [ ]:
video_file = files[0]
frame = load_video_frames_uniform(video_file)

In [ ]:
np.array(frame).shape

(30, 370, 656, 3)

In [ ]:
import os
DATA_PATH = "/content/drive/My Drive/Final Year Project/Data/Processed"
os.makedirs(DATA_PATH, exist_ok=True)
sequence_length = 30

In [ ]:
from tqdm.notebook import tqdm


with mp_holistic.Holistic(min_detection_confidence=0.5,
                          min_tracking_confidence=0.5) as holistic:

    for video_file in tqdm(files, desc="Processing videos"):
        file_name = os.path.splitext(os.path.basename(video_file))[0]
        split_cat = reference[reference['video_id'] == file_name]['split'].values.item()
        split_path = os.path.join(DATA_PATH, split_cat)
        os.makedirs(split_path, exist_ok=True)
        if os.path.exists(os.path.join(split_path, f"{file_name}.npy")):
            continue

        frames = load_video_frames_uniform(video_file, sequence_length)
        frames_keypoints = []

        # tqdm for frame processing
        for frame in tqdm(frames, desc=f"Frames ({file_name})", leave=False):
            image, results = mediapipe_detection(frame, holistic)
            keypoints = extract_keypoints(results)
            frames_keypoints.append(keypoints)

        # Convert to NumPy array
        frames_array = np.array(frames_keypoints)

        # Save as .npy
        save_path = os.path.join(split_path, f"{file_name}.npy")
        np.save(save_path, frames_array)

    reference.to_csv(os.path.join(DATA_PATH, "reference.csv"), index=False)


Processing videos:   0%|          | 0/11980 [00:00<?, ?it/s]

Frames (03607):   0%|          | 0/30 [00:00<?, ?it/s]

Frames (04007):   0%|          | 0/30 [00:00<?, ?it/s]

Frames (03756):   0%|          | 0/30 [00:00<?, ?it/s]

Frames (04076):   0%|          | 0/30 [00:00<?, ?it/s]

...

Frames (05734):   0%|          | 0/30 [00:00<?, ?it/s]

Frames (05988):   0%|          | 0/30 [00:00<?, ?it/s]

Frames (05967):   0%|          | 0/30 [00:00<?, ?it/s]

Frames (05962):   0%|          | 0/30 [00:00<?, ?it/s]

Frames (05925):   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:

frame_array.shape

NameError: name 'frame_array' is not defined